# Pipeline: Correction using Lapa LLM

This pipeline corrects zero-shot translations from OPUS-MT and mBART. To do that we ask Lapa LLM to rewrite target
sentences in way that would adhere to the source gender of a sentence.

In [1]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="lapa-llm/lapa-v0.1.2-instruct-GGUF",
	filename="lapa-v0.1.2-instruct-Q4_K_M.gguf",
)

/Users/pivanovs/.pyenv/versions/mt-proj/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
llama_model_load_from_file_impl: using device Metal (Apple M2 Max) - 25558 MiB free
llama_model_loader: loaded meta data with 42 key-value pairs and 626 tensors from /Users/pivanovs/.cache/huggingface/hub/models--lapa-llm--lapa-v0.1.2-instruct-GGUF/snapshots/1a30af6ad6341770e84e0eb51c27d86b6bf60b41/./lapa-v0.1.2-instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.n

In [2]:
def ensure_desired_gender(sentence: str, target_gender: str):
    editing_uk_texts = [
        {"role": "system", "content": "You are a linguistic assistant for Ukrainian text editing."},
        {"role": "user", "content": (
            f"Edit the following Ukrainian sentence by ensuring the occupation is in the {target_gender} form, "
            f"keeping the structure of the sentence unchanged:\n{sentence}"
        )}
    ]
    output = llm.create_chat_completion(messages=editing_uk_texts)
    corrected_sentence = output['choices'][0]['message']['content']
    return corrected_sentence
    

ensure_desired_gender(sentence='Я -- чудовий механік', target_gender='female')

llama_perf_context_print:        load time =    2761.05 ms
llama_perf_context_print: prompt eval time =    2760.80 ms /    48 tokens (   57.52 ms per token,    17.39 tokens per second)
llama_perf_context_print:        eval time =     300.38 ms /     4 runs   (   75.10 ms per token,    13.32 tokens per second)
llama_perf_context_print:       total time =    3063.26 ms /    52 tokens
llama_perf_context_print:    graphs reused =          3


'Я -- чудова механіка'

## mBART

In [4]:
import pandas as pd
from tqdm import tqdm


tqdm.pandas()

df = pd.read_csv('../data/mbart.zeroshot.translations.tsv', index_col=None, sep='\t')
df['mbart_llm_corrected'] = df.progress_apply(lambda r: ensure_desired_gender(r['mbart_hypothesis'], r['source_gender']), axis=1)
df.head()

  0%|          | 0/810 [00:00<?, ?it/s]Llama.generate: 27 prefix-match hit, remaining 22 prompt tokens to eval
llama_perf_context_print:        load time =    2761.05 ms
llama_perf_context_print: prompt eval time =     929.69 ms /    22 tokens (   42.26 ms per token,    23.66 tokens per second)
llama_perf_context_print:        eval time =     382.60 ms /     5 runs   (   76.52 ms per token,    13.07 tokens per second)
llama_perf_context_print:       total time =    1314.50 ms /    27 tokens
llama_perf_context_print:    graphs reused =          4
  0%|          | 2/810 [00:01<08:51,  1.52it/s]Llama.generate: 40 prefix-match hit, remaining 9 prompt tokens to eval
llama_perf_context_print:        load time =    2761.05 ms
llama_perf_context_print: prompt eval time =     657.76 ms /     9 tokens (   73.08 ms per token,    13.68 tokens per second)
llama_perf_context_print:        eval time =     384.24 ms /     5 runs   (   76.85 ms per token,    13.01 tokens per second)
llama_perf_context_

,template_type,occupation,source,source_gender,mbart_hypothesis,opusmt_llm_corrected
id,,,,,,
1,1,bartender,I used to work as a bartender.,neutral,Я працював барменом.,Я працював барменом.
2,2,bartender,I am a poor bartender.,neutral,Я бідний бармен.,Я бідний бармен.
3,3,bartender,I am a nice bartender.,neutral,Я хороший бармен.,Я хороший бармен.
4,4,bartender,The bartender called me today.,neutral,Бармен зателефонував мені сьогодні.,Бармен зателефонував мені сьогодні.
5,5,bartender,Anne is a bartender.,female,Анна - бармен.,Анна - барменка.


In [6]:
df.to_csv("../data/mbart.llm_corrected.translations.tsv", sep="\t", index=False)

## OPUS-MT

In [8]:
df = pd.read_csv('../data/opusmt.zeroshot.translations.tsv', index_col='id', sep='\t')
df['opusmt_llm_corrected'] = df.progress_apply(lambda r: ensure_desired_gender(r['opusmt_hypothesis'], r['source_gender']), axis=1)
df.head()
df.to_csv("../data/opusmt.llm_corrected.translations.tsv", sep="\t", index=False)


  0%|          | 0/810 [00:00<?, ?it/s]Llama.generate: 40 prefix-match hit, remaining 10 prompt tokens to eval
llama_perf_context_print:        load time =    2761.05 ms
llama_perf_context_print: prompt eval time = 1143480.43 ms /    11 tokens (103952.77 ms per token,     0.01 tokens per second)
llama_perf_context_print:        eval time =     468.83 ms /     6 runs   (   78.14 ms per token,    12.80 tokens per second)
llama_perf_context_print:       total time =    1021.38 ms /    17 tokens
llama_perf_context_print:    graphs reused =          5
  0%|          | 2/810 [00:01<06:53,  1.95it/s]Llama.generate: 40 prefix-match hit, remaining 9 prompt tokens to eval
llama_perf_context_print:        load time =    2761.05 ms
llama_perf_context_print: prompt eval time =     845.36 ms /     9 tokens (   93.93 ms per token,    10.65 tokens per second)
llama_perf_context_print:        eval time =     387.67 ms /     5 runs   (   77.53 ms per token,    12.90 tokens per second)
llama_perf_context